# NJ Property Tax & Equalization - Exploratory Analysis

**Data Sources:**
- `nj_property_tax`: 15,849 rows - Municipal property tax levies and collections
- `nj_equalization`: 13,565 rows - Property value equalization data

**Objectives:**
1. Understand property tax rates across NJ municipalities
2. Analyze tax burden variations by county and municipality
3. Examine equalization ratios and assessment practices
4. Identify highest/lowest tax jurisdictions
5. Explore temporal trends in tax rates and collections

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Connect to database
db_path = Path('../data/db/nj_pipeline.duckdb')
conn = duckdb.connect(str(db_path), read_only=True)
print(f"Connected to: {db_path}")

Connected to: ../data/db/nj_pipeline.duckdb


## 1. Data Overview

In [ ]:
# Schema information
print("=" * 60)
print("NJ PROPERTY TAX SCHEMA")
print("=" * 60)
display(conn.execute("DESCRIBE nj_property_tax").df())

print("\n" + "=" * 60)
print("NJ EQUALIZATION SCHEMA")
print("=" * 60)
display(conn.execute("DESCRIBE nj_equalization").df())

In [ ]:
# Sample data
print("\nSample Property Tax Data:")
display(conn.execute("""
    SELECT * FROM nj_property_tax 
    ORDER BY year DESC 
    LIMIT 5
""").df())

print("\nSample Equalization Data:")
display(conn.execute("""
    SELECT * FROM nj_equalization 
    ORDER BY year DESC 
    LIMIT 5
""").df())

In [ ]:
# Data coverage
tax_coverage = conn.execute("""
    SELECT 
        MIN(year) as earliest_year,
        MAX(year) as latest_year,
        COUNT(DISTINCT year) as num_years,
        COUNT(DISTINCT municipality) as num_municipalities,
        COUNT(DISTINCT county) as num_counties,
        COUNT(*) as total_records
    FROM nj_property_tax
""").df()

eq_coverage = conn.execute("""
    SELECT 
        MIN(year) as earliest_year,
        MAX(year) as latest_year,
        COUNT(DISTINCT year) as num_years,
        COUNT(DISTINCT municipality) as num_municipalities,
        COUNT(*) as total_records
    FROM nj_equalization
""").df()

print("\nProperty Tax Data Coverage:")
display(tax_coverage)

print("\nEqualization Data Coverage:")
display(eq_coverage)

## 2. Property Tax Analysis - Latest Year

In [ ]:
# Get latest year data
latest_tax_year = conn.execute("SELECT MAX(year) FROM nj_property_tax").fetchone()[0]
print(f"\nAnalyzing data for year: {latest_tax_year}")

latest_tax = conn.execute(f"""
    SELECT * FROM nj_property_tax
    WHERE year = {latest_tax_year}
    ORDER BY general_tax_rate DESC
""").df()

print(f"\nMunicipalities in {latest_tax_year}: {len(latest_tax)}")

In [ ]:
# Check available columns
print("\nAvailable columns:")
for col in latest_tax.columns:
    print(f"  • {col}")

# Identify key tax rate columns
rate_cols = [col for col in latest_tax.columns if 'rate' in col.lower()]
levy_cols = [col for col in latest_tax.columns if 'levy' in col.lower()]
assessed_cols = [col for col in latest_tax.columns if 'assessed' in col.lower() or 'value' in col.lower()]

print(f"\nRate columns: {rate_cols}")
print(f"Levy columns: {levy_cols}")
print(f"Assessment columns: {assessed_cols}")

In [ ]:
# Summary statistics for tax rates
if rate_cols:
    print(f"\nProperty Tax Rate Statistics ({latest_tax_year}):")
    display(latest_tax[rate_cols].describe().round(3))

In [ ]:
# Highest and lowest tax rate municipalities
if 'general_tax_rate' in latest_tax.columns and 'municipality' in latest_tax.columns:
    print(f"\nTop 20 Highest Tax Rate Municipalities ({latest_tax_year}):")
    display_cols = ['municipality', 'county', 'general_tax_rate'] if 'county' in latest_tax.columns else ['municipality', 'general_tax_rate']
    display(latest_tax.nlargest(20, 'general_tax_rate')[display_cols])
    
    print(f"\nTop 20 Lowest Tax Rate Municipalities ({latest_tax_year}):")
    display(latest_tax.nsmallest(20, 'general_tax_rate')[display_cols])

In [ ]:
# Tax rate distribution
if 'general_tax_rate' in latest_tax.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(latest_tax['general_tax_rate'].dropna(), bins=50, edgecolor='black', alpha=0.7)
    axes[0].axvline(latest_tax['general_tax_rate'].median(), color='red', linestyle='--', 
                   label=f'Median: {latest_tax["general_tax_rate"].median():.3f}')
    axes[0].set_xlabel('General Tax Rate')
    axes[0].set_ylabel('Number of Municipalities')
    axes[0].set_title(f'Distribution of Property Tax Rates ({latest_tax_year})')
    axes[0].legend()
    axes[0].grid(axis='y', alpha=0.3)
    
    # Box plot
    axes[1].boxplot(latest_tax['general_tax_rate'].dropna())
    axes[1].set_ylabel('General Tax Rate')
    axes[1].set_title(f'Tax Rate Distribution ({latest_tax_year})')
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 3. County-Level Analysis

In [ ]:
# County aggregates
if 'county' in latest_tax.columns and 'general_tax_rate' in latest_tax.columns:
    county_tax = latest_tax.groupby('county').agg({
        'general_tax_rate': ['mean', 'median', 'min', 'max', 'count']
    }).round(4)
    
    county_tax.columns = ['Avg Rate', 'Median Rate', 'Min Rate', 'Max Rate', 'Num Munis']
    county_tax = county_tax.sort_values('Median Rate', ascending=False)
    
    print(f"\nProperty Tax Rates by County ({latest_tax_year}):")
    display(county_tax)

In [ ]:
# Visualize county tax rates
if 'county' in latest_tax.columns and 'general_tax_rate' in latest_tax.columns:
    fig, ax = plt.subplots(figsize=(12, 10))
    
    county_tax_sorted = county_tax.sort_values('Median Rate')
    ax.barh(county_tax_sorted.index, county_tax_sorted['Median Rate'], alpha=0.7)
    ax.set_xlabel('Median Tax Rate')
    ax.set_title(f'Median Property Tax Rate by County ({latest_tax_year})')
    ax.grid(axis='x', alpha=0.3)
    
    for i, (county, value) in enumerate(zip(county_tax_sorted.index, county_tax_sorted['Median Rate'])):
        ax.text(value, i, f' {value:.3f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## 4. Equalization Analysis

In [ ]:
# Get latest equalization year
latest_eq_year = conn.execute("SELECT MAX(year) FROM nj_equalization").fetchone()[0]
print(f"\nAnalyzing equalization data for year: {latest_eq_year}")

latest_eq = conn.execute(f"""
    SELECT * FROM nj_equalization
    WHERE year = {latest_eq_year}
""").df()

print(f"\nMunicipalities: {len(latest_eq)}")
print("\nAvailable columns:")
for col in latest_eq.columns:
    print(f"  • {col}")

In [ ]:
# Check for equalization ratio column
eq_ratio_cols = [col for col in latest_eq.columns if 'ratio' in col.lower() or 'equalization' in col.lower()]
print(f"\nEqualization ratio columns: {eq_ratio_cols}")

if eq_ratio_cols:
    primary_eq_col = eq_ratio_cols[0]
    print(f"\nEqualization Ratio Statistics ({latest_eq_year}):")
    print(latest_eq[primary_eq_col].describe())
    
    # Municipalities with highest/lowest equalization ratios
    if 'municipality' in latest_eq.columns:
        print(f"\nHighest Equalization Ratios:")
        display(latest_eq.nlargest(10, primary_eq_col)[['municipality', primary_eq_col]])
        
        print(f"\nLowest Equalization Ratios:")
        display(latest_eq.nsmallest(10, primary_eq_col)[['municipality', primary_eq_col]])

In [ ]:
# Equalization ratio distribution
if eq_ratio_cols:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    eq_data = latest_eq[primary_eq_col].dropna()
    ax.hist(eq_data, bins=50, edgecolor='black', alpha=0.7, color='green')
    ax.axvline(eq_data.median(), color='red', linestyle='--', 
               label=f'Median: {eq_data.median():.2f}')
    ax.axvline(100, color='blue', linestyle=':', 
               label='Perfect Assessment (100)', alpha=0.7)
    ax.set_xlabel('Equalization Ratio')
    ax.set_ylabel('Number of Municipalities')
    ax.set_title(f'Distribution of Equalization Ratios ({latest_eq_year})\n(Ratio of assessed value to true market value)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Interpretation
    over_100 = len(eq_data[eq_data > 100])
    under_100 = len(eq_data[eq_data < 100])
    at_100 = len(eq_data[eq_data == 100])
    
    print(f"\nEqualization Ratio Distribution:")
    print(f"  Over-assessed (>100):  {over_100} ({over_100/len(eq_data)*100:.1f}%)")
    print(f"  Perfect (=100):        {at_100} ({at_100/len(eq_data)*100:.1f}%)")
    print(f"  Under-assessed (<100): {under_100} ({under_100/len(eq_data)*100:.1f}%)")

## 5. Time Series Analysis

In [ ]:
# Tax rate trends over time
if 'general_tax_rate' in conn.execute("SELECT * FROM nj_property_tax LIMIT 1").df().columns:
    tax_trends = conn.execute("""
        SELECT 
            year,
            AVG(general_tax_rate) as avg_rate,
            MEDIAN(general_tax_rate) as median_rate,
            MIN(general_tax_rate) as min_rate,
            MAX(general_tax_rate) as max_rate,
            COUNT(*) as num_municipalities
        FROM nj_property_tax
        GROUP BY year
        ORDER BY year
    """).df()
    
    print("\nTax Rate Trends Over Time:")
    display(tax_trends)
    
    # Plot trends
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(tax_trends['year'], tax_trends['median_rate'], marker='o', linewidth=2, label='Median')
    ax.plot(tax_trends['year'], tax_trends['avg_rate'], marker='s', linewidth=2, label='Average', alpha=0.7)
    ax.fill_between(tax_trends['year'], tax_trends['min_rate'], tax_trends['max_rate'], alpha=0.2, label='Min-Max Range')
    ax.set_xlabel('Year')
    ax.set_ylabel('General Tax Rate')
    ax.set_title('NJ Property Tax Rate Trends')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 6. Tax Burden Analysis

In [ ]:
# Calculate effective tax on median home value
# Assuming NJ median home value from Zillow data
nj_median_home = 450000  # Example value, adjust based on Zillow data

if 'general_tax_rate' in latest_tax.columns:
    latest_tax['annual_tax_on_median_home'] = (latest_tax['general_tax_rate'] * nj_median_home / 100).round(0)
    
    print(f"\nEstimated Annual Tax on ${nj_median_home:,} Home:")
    print(f"  Minimum:  ${latest_tax['annual_tax_on_median_home'].min():,.0f}")
    print(f"  Median:   ${latest_tax['annual_tax_on_median_home'].median():,.0f}")
    print(f"  Maximum:  ${latest_tax['annual_tax_on_median_home'].max():,.0f}")
    print(f"  Range:    ${latest_tax['annual_tax_on_median_home'].max() - latest_tax['annual_tax_on_median_home'].min():,.0f}")
    
    # Show municipalities with highest/lowest estimated taxes
    if 'municipality' in latest_tax.columns:
        print(f"\nHighest Tax Municipalities (on ${nj_median_home:,} home):")
        display(latest_tax.nlargest(10, 'annual_tax_on_median_home')[['municipality', 'county', 'general_tax_rate', 'annual_tax_on_median_home']])
        
        print(f"\nLowest Tax Municipalities (on ${nj_median_home:,} home):")
        display(latest_tax.nsmallest(10, 'annual_tax_on_median_home')[['municipality', 'county', 'general_tax_rate', 'annual_tax_on_median_home']])

In [ ]:
# Visualize tax burden
if 'annual_tax_on_median_home' in latest_tax.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.hist(latest_tax['annual_tax_on_median_home'].dropna() / 1000, bins=50, 
            edgecolor='black', alpha=0.7, color='indianred')
    ax.axvline(latest_tax['annual_tax_on_median_home'].median() / 1000, 
               color='red', linestyle='--', 
               label=f'Median: ${latest_tax["annual_tax_on_median_home"].median()/1000:.1f}k')
    ax.set_xlabel('Estimated Annual Property Tax ($1000s)')
    ax.set_ylabel('Number of Municipalities')
    ax.set_title(f'Distribution of Estimated Annual Property Tax\n(Based on ${nj_median_home:,} home value)')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 7. Combined Tax & Equalization Analysis

In [ ]:
# Merge tax and equalization data
if 'municipality' in latest_tax.columns and 'municipality' in latest_eq.columns:
    combined = latest_tax.merge(
        latest_eq[['municipality'] + eq_ratio_cols], 
        on='municipality', 
        how='inner',
        suffixes=('_tax', '_eq')
    )
    
    print(f"\nCombined dataset: {len(combined)} municipalities")
    
    if eq_ratio_cols and 'general_tax_rate' in combined.columns:
        # Scatter plot: Tax rate vs Equalization ratio
        fig, ax = plt.subplots(figsize=(12, 8))
        
        ax.scatter(combined[primary_eq_col], combined['general_tax_rate'], alpha=0.5)
        ax.set_xlabel('Equalization Ratio')
        ax.set_ylabel('General Tax Rate')
        ax.set_title('Property Tax Rate vs Equalization Ratio')
        ax.axvline(100, color='red', linestyle='--', alpha=0.3, label='Perfect Assessment')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Correlation
        corr = combined[[primary_eq_col, 'general_tax_rate']].corr().iloc[0, 1]
        print(f"\nCorrelation between equalization ratio and tax rate: {corr:.3f}")

## 8. Key Findings Summary

In [ ]:
print("="*70)
print("KEY FINDINGS - NJ PROPERTY TAX & EQUALIZATION")
print("="*70)

print(f"\n1. DATA COVERAGE")
print(f"   • Tax data years: {tax_coverage['earliest_year'].iloc[0]} - {tax_coverage['latest_year'].iloc[0]}")
print(f"   • Municipalities: {tax_coverage['num_municipalities'].iloc[0]}")
print(f"   • Counties: {tax_coverage['num_counties'].iloc[0]}")

if 'general_tax_rate' in latest_tax.columns:
    print(f"\n2. TAX RATES ({latest_tax_year})")
    print(f"   • Median rate:  {latest_tax['general_tax_rate'].median():.3f}")
    print(f"   • Average rate: {latest_tax['general_tax_rate'].mean():.3f}")
    print(f"   • Lowest rate:  {latest_tax['general_tax_rate'].min():.3f}")
    print(f"   • Highest rate: {latest_tax['general_tax_rate'].max():.3f}")
    print(f"   • Range:        {latest_tax['general_tax_rate'].max() - latest_tax['general_tax_rate'].min():.3f}")

if 'annual_tax_on_median_home' in latest_tax.columns:
    print(f"\n3. TAX BURDEN (${nj_median_home:,} home)")
    print(f"   • Median annual tax: ${latest_tax['annual_tax_on_median_home'].median():,.0f}")
    print(f"   • Lowest annual tax: ${latest_tax['annual_tax_on_median_home'].min():,.0f}")
    print(f"   • Highest annual tax: ${latest_tax['annual_tax_on_median_home'].max():,.0f}")

if 'county' in latest_tax.columns and 'general_tax_rate' in latest_tax.columns:
    highest_county = county_tax.index[0]
    lowest_county = county_tax.index[-1]
    print(f"\n4. COUNTY VARIATIONS")
    print(f"   • Highest median rate: {highest_county} ({county_tax.loc[highest_county, 'Median Rate']:.3f})")
    print(f"   • Lowest median rate:  {lowest_county} ({county_tax.loc[lowest_county, 'Median Rate']:.3f})")

if eq_ratio_cols:
    eq_data = latest_eq[primary_eq_col].dropna()
    print(f"\n5. EQUALIZATION RATIOS ({latest_eq_year})")
    print(f"   • Median ratio: {eq_data.median():.2f}")
    print(f"   • Average ratio: {eq_data.mean():.2f}")
    print(f"   • Over-assessed (>100): {len(eq_data[eq_data > 100])} munis ({len(eq_data[eq_data > 100])/len(eq_data)*100:.1f}%)")
    print(f"   • Under-assessed (<100): {len(eq_data[eq_data < 100])} munis ({len(eq_data[eq_data < 100])/len(eq_data)*100:.1f}%)")

if len(tax_trends) > 1:
    first_year_rate = tax_trends.iloc[0]['median_rate']
    last_year_rate = tax_trends.iloc[-1]['median_rate']
    pct_change = (last_year_rate - first_year_rate) / first_year_rate * 100
    print(f"\n6. TRENDS")
    print(f"   • Median rate in {tax_trends.iloc[0]['year']}: {first_year_rate:.3f}")
    print(f"   • Median rate in {tax_trends.iloc[-1]['year']}: {last_year_rate:.3f}")
    print(f"   • Change: {pct_change:+.1f}%")

print("\n" + "="*70)

In [ ]:
conn.close()
print("\nAnalysis complete. Database connection closed.")